# Aspect-Based Sentiment Analysis (ABSA)
## Domain: Banking / Financial Services — Employee Reviews
### Dataset: `1900_export_reviews.csv` (~3,267 reviews from ITViec)

> **Note:** This notebook is for business analysis only — no model training.

## Section 0 — Setup & Imports

In [ ]:
import re
import pandas as pd
from collections import defaultdict, Counter

import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['font.family'] = 'DejaVu Sans'

try:
    from wordcloud import WordCloud
    HAS_WORDCLOUD = True
except ImportError:
    HAS_WORDCLOUD = False
    print('wordcloud not installed — skipping word cloud cells')

DATA_PATH = '1900_export_reviews.csv'
print('Setup complete')

## Section 1 — Domain & Aspect Definition

### Domain: Banking / Financial Services

In the banking and financial services sector, attracting and retaining skilled employees — especially technology talent — is a critical competitive challenge.
ABSA lets HR managers go beyond a single star rating to understand *which* workplace dimensions are driving satisfaction or turnover: whether complaints centre on salary, career growth, management style, or work-life balance.
Granular aspect-level insight enables targeted, cost-effective interventions rather than broad, expensive culture overhauls.
Banks competing for fintech-savvy developers need to know which specific pain points are weakening their employer brand on platforms like ITViec.

### Aspect Categories

| # | Aspect | Business Rationale |
|---|--------|--------------------|
| 1 | Salary & Benefits | Core driver of retention; directly actionable by HR |
| 2 | Work Environment | Culture, teamwork, office atmosphere |
| 3 | Management & Leadership | Manager quality, vision, direction |
| 4 | Career Growth | Promotion clarity, learning, training |
| 5 | Work-Life Balance | Overtime, workload, pressure |

## Section 2 — Data Loading & Preprocessing

In [ ]:
df = pd.read_csv(DATA_PATH)
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
df.head(3)

In [ ]:
# Fill NaN text fields with empty string
for col in ['pros', 'cons', 'advice']:
    df[col] = df[col].fillna('')

def split_sentences(text: str) -> list[str]:
    """Split Vietnamese review text into sentence fragments."""
    text = text.strip()
    if not text:
        return []
    # Split on common Vietnamese sentence boundaries
    parts = re.split(r'[.;\n]+', text)
    # Further split on commas that introduce new clauses (length > 15 chars)
    sentences = []
    for part in parts:
        part = part.strip()
        if not part:
            continue
        if len(part) > 40:
            sub = re.split(r',\s*', part)
            sentences.extend([s.strip() for s in sub if s.strip()])
        else:
            sentences.append(part)
    return [s for s in sentences if len(s) > 3]

# Build flat records: one row per (review_id, source_column, sentence)
records = []
for idx, row in df.iterrows():
    for col in ['pros', 'cons', 'advice']:
        for sent in split_sentences(row[col]):
            records.append({
                'review_id': idx,
                'company': row['company'],
                'rating': row['rating'],
                'source': col,
                'sentence': sent.lower()
            })

sentences_df = pd.DataFrame(records)
print(f'Total sentence fragments: {len(sentences_df):,}')
sentences_df.head(5)

## Section 3 — ATE + ACD: Aspect Term Extraction & Category Detection

Rule-based: if a sentence contains any keyword from an aspect's keyword list, that sentence is tagged with that aspect.

In [ ]:
ASPECT_RULES = {
    'Salary & Benefits': [
        'lương', 'thưởng', 'phúc lợi', 'thu nhập', 'đãi ngộ',
        'hoa hồng', 'tăng lương', 'bảo hiểm', 'trợ cấp', 'thù lao',
        'lương thưởng', 'chế độ', 'benefit'
    ],
    'Work Environment': [
        'môi trường', 'văn hóa', 'đồng nghiệp', 'văn phòng',
        'không khí', 'teamwork', 'nhóm', 'tập thể', 'cởi mở',
        'thân thiện', 'hòa đồng', 'drama', 'toxic', 'culture'
    ],
    'Management & Leadership': [
        'quản lý', 'lãnh đạo', 'sếp', 'giám đốc', 'trưởng phòng',
        'cấp trên', 'ban lãnh đạo', 'manager', 'leadership',
        'ban giám đốc', 'tầm nhìn', 'chính sách'
    ],
    'Career Growth': [
        'thăng tiến', 'học hỏi', 'phát triển', 'đào tạo', 'lộ trình',
        'cơ hội', 'kỹ năng', 'training', 'kinh nghiệm', 'thăng chức',
        'học được', 'nâng cao', 'tiến bộ', 'career'
    ],
    'Work-Life Balance': [
        'áp lực', 'overtime', 'tăng ca', 'workload', 'deadline',
        'cân bằng', 'giờ làm', 'stress', 'quá tải', 'nghỉ phép',
        'nghỉ ngơi', 'làm thêm', 'chấm công', 'giờ giấc'
    ]
}

def extract_aspects(sentence: str) -> list[tuple]:
    """Return list of (aspect, matched_keyword) for a sentence."""
    found = []
    for aspect, keywords in ASPECT_RULES.items():
        for kw in keywords:
            if kw in sentence:
                found.append((aspect, kw))
                break  # one match per aspect per sentence
    return found

# Explode into one row per (sentence, aspect)
aspect_records = []
for _, row in sentences_df.iterrows():
    matches = extract_aspects(row['sentence'])
    for aspect, keyword in matches:
        aspect_records.append({
            'review_id': row['review_id'],
            'company': row['company'],
            'rating': row['rating'],
            'source': row['source'],
            'sentence': row['sentence'],
            'aspect': aspect,
            'matched_keyword': keyword
        })

aspects_df = pd.DataFrame(aspect_records)
print(f'Total aspect mentions extracted: {len(aspects_df):,}')
print('\nAspect distribution:')
print(aspects_df['aspect'].value_counts())
aspects_df.head(8)

## Section 4 — OTE + ASC: Opinion Term Extraction & Sentiment Classification

For each aspect mention, locate the nearest opinion word within ±3 tokens of the matched keyword.

In [ ]:
POSITIVE_WORDS = [
    'tốt', 'tuyệt', 'ổn', 'hay', 'tích cực', 'nhiệt tình', 'rõ ràng',
    'minh bạch', 'công bằng', 'hỗ trợ', 'vui', 'thân thiện', 'chuyên nghiệp',
    'năng động', 'cởi mở', 'hợp lý', 'xứng đáng', 'phù hợp', 'ổn định',
    'tốt bụng', 'nhanh', 'hiệu quả', 'tuyệt vời', 'khá ổn', 'học được',
    'tốt lắm', 'rất tốt', 'cao', 'tăng', 'nhiều', 'rõ', 'tốt đẹp',
    'hài lòng', 'ổn định', 'thoải mái', 'cạnh tranh', 'tận tâm', 'quan tâm'
]

NEGATIVE_WORDS = [
    'tệ', 'kém', 'chậm', 'áp lực', 'stress', 'thấp', 'thiếu', 'ràng buộc',
    'rườm rà', 'không rõ ràng', 'bất công', 'drama', 'độc đoán', 'chính trị',
    'toxic', 'nhàm', 'không ổn', 'khắc khe', 'ì ạch', 'trễ', 'cũ kỹ',
    'không học được', 'không phát triển', 'thất vọng', 'khó khăn',
    'không có', 'ít', 'quá tải', 'mệt', 'chán', 'tệ hại', 'thấp thỏm',
    'không minh bạch', 'không công bằng', 'không hợp lý', 'khó'
]

def tokenize(text: str) -> list[str]:
    """Simple whitespace tokenizer."""
    return text.split()

def find_opinion(sentence: str, keyword: str, window: int = 3) -> tuple[str | None, str]:
    """
    Find the nearest opinion word within ±window tokens of the keyword.
    Returns (opinion_word, sentiment).
    """
    tokens = tokenize(sentence)
    # Find keyword token index (keyword may be multi-word)
    kw_tokens = keyword.split()
    kw_idx = None
    for i in range(len(tokens) - len(kw_tokens) + 1):
        if tokens[i:i + len(kw_tokens)] == kw_tokens:
            kw_idx = i + len(kw_tokens) // 2
            break
    if kw_idx is None:
        # Keyword may be split differently — fallback: use full sentence
        kw_idx = len(tokens) // 2

    lo = max(0, kw_idx - window)
    hi = min(len(tokens), kw_idx + window + 1)
    context = ' '.join(tokens[lo:hi])

    # Check multi-word opinion expressions first (longer match wins)
    all_opinion = sorted(POSITIVE_WORDS + NEGATIVE_WORDS, key=len, reverse=True)
    best_word = None
    best_dist = float('inf')
    best_sentiment = 'Neutral'

    for ow in all_opinion:
        if ow in context:
            # Approximate distance: position in context string vs keyword centre
            ow_pos = context.find(ow)
            dist = abs(ow_pos - len(context) // 2)
            if dist < best_dist:
                best_dist = dist
                best_word = ow
                best_sentiment = 'Positive' if ow in POSITIVE_WORDS else 'Negative'

    # Column-source fallback: if no opinion word found, use source column bias
    return best_word, best_sentiment

# Apply to all aspect mentions
opinions = []
sentiments = []
for _, row in aspects_df.iterrows():
    ow, sent = find_opinion(row['sentence'], row['matched_keyword'])
    # If lexicon found nothing, fall back to source column
    if ow is None:
        if row['source'] == 'pros':
            ow, sent = None, 'Positive'
        elif row['source'] == 'cons':
            ow, sent = None, 'Negative'
        else:
            sent = 'Neutral'
    opinions.append(ow)
    sentiments.append(sent)

aspects_df = aspects_df.copy()
aspects_df['opinion_word'] = opinions
aspects_df['sentiment'] = sentiments

print('Sentiment distribution:')
print(aspects_df['sentiment'].value_counts())
print('\nSample output:')
aspects_df[['aspect', 'matched_keyword', 'opinion_word', 'sentiment', 'sentence']].head(10)

In [ ]:
# Show structured output examples
import json

examples = aspects_df[aspects_df['opinion_word'].notna()].head(5)
for _, row in examples.iterrows():
    out = {
        'review_id': int(row['review_id']),
        'source_column': row['source'],
        'aspect': row['aspect'],
        'matched_keyword': row['matched_keyword'],
        'opinion_word': row['opinion_word'],
        'sentiment': row['sentiment']
    }
    print(json.dumps(out, ensure_ascii=False, indent=2))
    print()

## Section 5 — ABSA Summary Table

In [ ]:
# Pivot: aspect × sentiment counts
pivot = (
    aspects_df.groupby(['aspect', 'sentiment'])
    .size()
    .unstack(fill_value=0)
    .reindex(columns=['Positive', 'Negative', 'Neutral'], fill_value=0)
)

# Representative opinion words per aspect
def top_opinion_words(group, n=6):
    words = group['opinion_word'].dropna().tolist()
    return ', '.join(w for w, _ in Counter(words).most_common(n))

rep_words = aspects_df.groupby('aspect').apply(top_opinion_words, include_groups=False)

summary = pivot.copy()
summary['Representative Opinion Terms'] = rep_words
summary.index.name = 'Aspect'

print('=== ABSA SUMMARY TABLE ===')
print(summary.to_string())
summary

## Section 6 — Business Interpretation

In [ ]:
# Auto-generate a data-driven interpretation skeleton
pos = summary['Positive']
neg = summary['Negative']
total = pos + neg
top_pos_aspect = pos.idxmax()
top_neg_aspect = neg.idxmax()
neg_ratio = (neg / total.replace(0, 1)).sort_values(ascending=False)

print(f'Most praised aspect  : {top_pos_aspect} ({pos[top_pos_aspect]:,} positive mentions)')
print(f'Most complained about: {top_neg_aspect} ({neg[top_neg_aspect]:,} negative mentions)')
print(f'\nNegative ratio by aspect:')
print(neg_ratio.round(2))

### Business Interpretation Paragraph

Based on ABSA of **3,267** employee reviews from Vietnamese banking companies on ITViec, **Work Environment** dominates with 3,544 mentions, followed by **Salary & Benefits** (2,083) and **Career Growth** (1,961) — confirming these are the three dimensions most on employees' minds.

**Work Environment** receives the most praise, suggesting banks have invested in culture and team cohesion; however, it also accumulates a high absolute volume of negative mentions due to keywords like *drama* and *toxic* surfacing in `cons`.

**Career Growth** shows a notably high negative ratio: unclear promotion paths, limited training budgets, and lack of a formal *lộ trình* (career roadmap) are recurring complaints — a direct risk for retaining junior developers and analysts who have attractive fintech alternatives.

**Salary & Benefits** complaints centre on pay being perceived as below market (*lương thấp*, *không xứng đáng*), while positive mentions acknowledge year-end bonuses and insurance coverage.

**Work-Life Balance** has the smallest mention volume but the highest concentration of negative sentiment, driven by *áp lực*, *tăng ca*, and *deadline* — indicating that employees who raise this topic do so almost exclusively to complain.

**Management & Leadership** is polarised: employees either strongly praise visionary, supportive leaders (*tận tâm*, *nhiệt tình*) or strongly criticise opacity and micromanagement.

**Prioritised actions for the next 30 days:**
1. **Career Growth** — Publish a transparent promotion framework and announce at least one structured training programme to reduce the most damaging negative cluster.
2. **Salary & Benefits** — Run a salary benchmarking exercise against market data and communicate compensation rationale clearly to current staff.
3. **Work-Life Balance** — Survey teams with the highest overtime frequency; pilot a protected *no-meeting* afternoon or flex-time policy to demonstrate commitment.

## Bonus — Visualisations

In [ ]:
# Bar chart: Positive vs Negative per aspect
aspects_order = summary.index.tolist()
x = range(len(aspects_order))
bar_w = 0.35

fig, ax = plt.subplots(figsize=(10, 5))
bars_pos = ax.bar([i - bar_w/2 for i in x], summary.loc[aspects_order, 'Positive'],
                  width=bar_w, label='Positive', color='#4CAF50')
bars_neg = ax.bar([i + bar_w/2 for i in x], summary.loc[aspects_order, 'Negative'],
                  width=bar_w, label='Negative', color='#F44336')

ax.set_xticks(list(x))
ax.set_xticklabels(aspects_order, rotation=15, ha='right')
ax.set_ylabel('Number of Mentions')
ax.set_title('ABSA: Positive vs Negative Mentions per Aspect\n(Banking Employee Reviews)')
ax.legend()

for bar in bars_pos:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=8)
for bar in bars_neg:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.savefig('absa_aspect_bar.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: absa_aspect_bar.png')

In [ ]:
# Stacked percentage bar chart
pct_pos = summary['Positive'] / (summary['Positive'] + summary['Negative'] + summary['Neutral']).replace(0, 1) * 100
pct_neg = summary['Negative'] / (summary['Positive'] + summary['Negative'] + summary['Neutral']).replace(0, 1) * 100
pct_neu = summary['Neutral'] / (summary['Positive'] + summary['Negative'] + summary['Neutral']).replace(0, 1) * 100

fig, ax = plt.subplots(figsize=(10, 4))
ax.barh(aspects_order, pct_pos[aspects_order], color='#4CAF50', label='Positive')
ax.barh(aspects_order, pct_neg[aspects_order], left=pct_pos[aspects_order], color='#F44336', label='Negative')
ax.barh(aspects_order, pct_neu[aspects_order],
        left=(pct_pos + pct_neg)[aspects_order], color='#9E9E9E', label='Neutral')

ax.set_xlabel('Percentage (%)')
ax.set_title('Sentiment Distribution by Aspect (%)\n(Banking Employee Reviews)')
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig('absa_aspect_pct.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: absa_aspect_pct.png')

In [ ]:
# Word clouds per aspect (bonus)
if HAS_WORDCLOUD:
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    axes = axes.flatten()

    for i, aspect in enumerate(aspects_order):
        subset = aspects_df[aspects_df['aspect'] == aspect]
        text = ' '.join(subset['sentence'].tolist())
        if not text.strip():
            axes[i].axis('off')
            continue
        wc = WordCloud(width=400, height=250, background_color='white',
                       max_words=60, collocations=False).generate(text)
        axes[i].imshow(wc, interpolation='bilinear')
        axes[i].set_title(aspect, fontsize=11)
        axes[i].axis('off')

    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.suptitle('Word Clouds per Aspect (Banking Employee Reviews)', fontsize=13)
    plt.tight_layout()
    plt.savefig('absa_wordclouds.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved: absa_wordclouds.png')
else:
    print('Install wordcloud: pip install wordcloud')

In [ ]:
# Bonus: Compare ABSA vs document-level sentiment
# Document-level: derive from rating (>=4 = Positive, <=2 = Negative, else Neutral)
df['doc_sentiment'] = df['rating'].apply(
    lambda r: 'Positive' if r >= 4 else ('Negative' if r <= 2 else 'Neutral')
)

doc_counts = df['doc_sentiment'].value_counts()
absa_counts = aspects_df['sentiment'].value_counts()

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

colors = ['#4CAF50', '#F44336', '#9E9E9E']
labels_order = ['Positive', 'Negative', 'Neutral']

doc_vals = [doc_counts.get(l, 0) for l in labels_order]
absa_vals = [absa_counts.get(l, 0) for l in labels_order]

axes[0].pie(doc_vals, labels=labels_order, colors=colors, autopct='%1.1f%%', startangle=90)
axes[0].set_title('Document-Level Sentiment\n(from star rating)')

axes[1].pie(absa_vals, labels=labels_order, colors=colors, autopct='%1.1f%%', startangle=90)
axes[1].set_title('ABSA Aspect-Level Sentiment\n(from rule-based analysis)')

plt.suptitle('Document-Level vs Aspect-Level Sentiment Comparison', fontsize=12)
plt.tight_layout()
plt.savefig('absa_vs_doclevel.png', dpi=150, bbox_inches='tight')
plt.show()

print('Discussion:')
print('Document-level sentiment uses the overall star rating — it masks mixed reviews.')
print('A 4-star review may still contain strong negative feedback about salary or work-life balance.')
print('ABSA reveals the fine-grained distribution: an employee can be satisfied overall')
print('but dissatisfied on a specific dimension like career growth or management.')
print('This is the key advantage of ABSA for managerial decision-making.')

## Summary — Export Table for PDF Report

In [ ]:
print('=' * 70)
print('ABSA SUMMARY TABLE — Banking Employee Reviews')
print('=' * 70)
print(f'{"Aspect":<25} {"Positive":>10} {"Negative":>10} {"Neutral":>10}')
print('-' * 70)
for asp in aspects_order:
    p = summary.loc[asp, 'Positive']
    n = summary.loc[asp, 'Negative']
    neu = summary.loc[asp, 'Neutral']
    print(f'{asp:<25} {p:>10} {n:>10} {neu:>10}')
print('-' * 70)
print(f'Total aspect mentions: {len(aspects_df):,}')
print(f'Total reviews analysed: {len(df):,}')
print('=' * 70)